# SIS Risk Propagation on Empirical Financial Network
Based on **Chen et al. (2023)** — *Applied Sciences* 13, 1129.

Pipeline :
1. Réseau PMFG filtré sur la matrice de corrélation empirique |C_ij|
2. Modèle SIS sur le réseau → état stationnaire x*
3. Matrice de réponse linéaire G = (-J)⁻¹ normalisée
4. Flow de propagation F_i (knockout de nœud)
5. Temps de réponse τ_i
6. Effet degree-driven : F_i ∼ S_i^ω, τ_i ∼ S_i^θ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.integrate import solve_ivp
from scipy.linalg import inv
from functions import load_data, correlation

plt.rcParams['figure.dpi'] = 110
rng = np.random.default_rng(42)

In [ ]:
data = load_data(['stock'], log_returns=True, sort_by_sector=True)
N, T = data.shape[1], data.shape[0]
asset_names = data.columns.tolist()

# Matrice de corrélation contemporanée (lag=0)
C_emp = correlation(data.values, lag=0)
C_abs = np.abs(C_emp)
np.fill_diagonal(C_abs, 0.0)

print(f'N = {N},  T = {T}')
print(f'Mean |C_ij| off-diag = {C_abs[C_abs > 0].mean():.3f}')

## 1. Réseau PMFG

Le **Planar Maximally Filtered Graph** (Tumminello 2005) ajoute les arêtes par ordre décroissant de |C_ij| en
maintenant la planarité du graphe. Il conserve exactement `3(N-2)` arêtes — beaucoup plus dense que le MST
(N-1 arêtes) mais toujours filtré. Ici N=146 → 432 arêtes.

In [ ]:
import time

def build_pmfg(C_abs):
    """Planar Maximally Filtered Graph (Tumminello et al. 2005)."""
    n = C_abs.shape[0]
    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(n))
    edges = sorted(
        [(i, j, C_abs[i, j]) for i in range(n) for j in range(i + 1, n)],
        key=lambda e: -e[2]
    )
    max_edges = 3 * (n - 2)
    A = np.zeros((n, n))
    added = 0
    for k, (i, j, w) in enumerate(edges):
        if added >= max_edges:
            break
        G_nx.add_edge(i, j)
        if nx.check_planarity(G_nx)[0]:
            A[i, j] = A[j, i] = w
            added += 1
        else:
            G_nx.remove_edge(i, j)
        if k % 2000 == 0 and k > 0:
            print(f'  {k}/{len(edges)} arêtes testées, {added} acceptées...')
    print(f'PMFG : {added} arêtes (attendu {max_edges})')
    return A, G_nx

t0 = time.time()
A_pmfg, G_nx = build_pmfg(C_abs)
print(f'Construit en {time.time() - t0:.1f}s')

S   = A_pmfg.sum(axis=1)            # degré pondéré
deg = (A_pmfg > 0).sum(axis=1)      # degré topologique

print(f'Degré : mean = {deg.mean():.1f}, max = {deg.max()}')
print(f'S_i   : mean = {S.mean():.4f}, max = {S.max():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.heatmap(A_pmfg, cmap='YlOrRd', ax=axes[0], square=True,
            xticklabels=False, yticklabels=False, cbar=True)
axes[0].set_title(f'PMFG — {(A_pmfg > 0).sum() // 2} arêtes', fontsize=12)

axes[1].hist(deg, bins=20, color='#2166ac', edgecolor='white')
axes[1].set_xlabel('Degré'); axes[1].set_ylabel('# actifs')
axes[1].set_title('Distribution des degrés'); axes[1].grid(alpha=0.3)

axes[2].hist(S, bins=30, color='#b2182b', edgecolor='white')
axes[2].set_xlabel('Degré pondéré S_i'); axes[2].set_ylabel('# actifs')
axes[2].set_title('Distribution S_i'); axes[2].grid(alpha=0.3)

plt.suptitle('Réseau PMFG — 146 actifs US', fontsize=13)
plt.tight_layout(); plt.show()

## 2. Modèle SIS — état stationnaire

`dx_i/dt = -B·x_i + R·(1-x_i)·Σ_j A_ij·x_j`

- x_i(t) : probabilité d'infection de l'actif i
- R = taux d'infection, B = taux de guérison  
- Seuil épidémique : R/B > 1/λ_max(A_PMFG)

On intègre jusqu'à convergence (t_max=200, x0=0.5 partout).

In [ ]:
R_sis = 1.0
B_sis = 1.0

lam_max_pmfg = float(np.linalg.eigvalsh(A_pmfg)[-1])
threshold = B_sis / (R_sis * lam_max_pmfg)
print(f'λ_max(A_PMFG) = {lam_max_pmfg:.3f}')
print(f'Seuil épidémique 1/λ_max = {threshold:.4f}  → R/B = {R_sis/B_sis:.1f}  '
      f'{"(au-dessus, épidémie)" if R_sis/B_sis > threshold else "(en dessous, extinction)"}')

def sis_ode(t, x, A, R, B):
    return -B * x + R * (1 - x) * (A @ x)

x0 = np.full(N, 0.5)
sol = solve_ivp(sis_ode, [0, 200], x0, args=(A_pmfg, R_sis, B_sis),
                method='RK45', rtol=1e-8, atol=1e-10)
x_ss = sol.y[:, -1]
x_ss = np.clip(x_ss, 1e-6, 1 - 1e-6)

print(f'État stationnaire x*: mean = {x_ss.mean():.4f}, '
      f'min = {x_ss.min():.4f}, max = {x_ss.max():.4f}')

## 3. Matrice de réponse linéaire G

Jacobien au point fixe :
- J_ii = -B - R·(A·x*)_i
- J_ij = R·(1-x*_i)·A_ij  (i ≠ j)

Réponse linéaire : G_raw = (-J)⁻¹  
Normalisation log-log : G_mn = |G_raw_mn · x*_n / x*_m|

In [ ]:
Ax_ss = A_pmfg @ x_ss

# Jacobien
J = R_sis * np.diag(1 - x_ss) @ A_pmfg
np.fill_diagonal(J, -(B_sis + R_sis * Ax_ss))

eigs = np.linalg.eigvals(J).real
print(f'Eigenvalues de J : max = {eigs.max():.4f}  (doit être < 0 pour stabilité)')

# Matrice de réponse linéaire
G_raw = inv(-J)
G = np.abs(G_raw * x_ss[np.newaxis, :] / x_ss[:, np.newaxis])
G_col_sum = G.sum(axis=0)   # Σ_m G_mn pour chaque source n

print(f'G : mean = {G.mean():.4f}, max = {G.max():.4f}')

## 4. Flow de propagation F_i

**Approche knockout** : on supprime le nœud i, on recalcule G_sub, et on mesure quelle fraction du flow total passe par i.

`F_i = (1/N) Σ_n [(Σ_m G_mn - Σ_m G_sub_mn^{-i}) / Σ_m G_mn]`

Approximation : le retrait du nœud i modifie légèrement x* — on utilise x*_sub ≈ x*[idx] pour éviter N re-simulations.
Le calcul est vectorisé : N inversions de matrices (N-1)×(N-1) en un seul appel `np.linalg.inv`.

In [ ]:
all_idx = np.arange(N)

# Construire les N matrices J_sub d'un coup — shape (N, N-1, N-1)
print('Construction des J_sub...', end=' ')
J_subs   = np.stack([J[np.ix_(np.delete(all_idx, i),
                               np.delete(all_idx, i))] for i in range(N)])
x_subs   = np.stack([np.delete(x_ss, i) for i in range(N)])
print('OK')

# Inversion batch
print('Inversion batch (-J_sub)...', end=' ')
G_sub_raws = np.linalg.inv(-J_subs)                        # (N, N-1, N-1)
eps = 1e-10
G_subs = np.abs(G_sub_raws
                * x_subs[:, np.newaxis, :]
                / (x_subs[:, :, np.newaxis] + eps))         # (N, N-1, N-1)
G_sub_col_sums = G_subs.sum(axis=1)                         # (N, N-1)
print('OK')

# F_i
F_i = np.zeros(N)
for i in range(N):
    denom = np.delete(G_col_sum, i)                          # (N-1,)
    numer = denom - G_sub_col_sums[i]
    F_i[i] = np.mean(numer / (denom + eps))

print(f'F_i : mean = {F_i.mean():.4f}, max = {F_i.max():.4f}')
top5 = np.argsort(F_i)[::-1][:5]
for k in top5:
    print(f'  {asset_names[k]:10s}  S_i = {S[k]:.4f}  F_i = {F_i[k]:.4f}')

## 5. Temps de réponse τ_i

Approximation diagonale : τ_i ≈ ln(2) / (-J_ii) = ln(2) / (B + R·(A·x*)_i)

Cela donne τ_i ∝ 1/S_i_eff (effet degree-driven inverse).

In [ ]:
tau_i = np.log(2) / (B_sis + R_sis * Ax_ss)

print(f'τ_i : mean = {tau_i.mean():.4f}, min = {tau_i.min():.4f}, max = {tau_i.max():.4f}')
fast5 = np.argsort(tau_i)[:5]
for k in fast5:
    print(f'  {asset_names[k]:10s}  S_i = {S[k]:.4f}  τ_i = {tau_i[k]:.4f}')

## 6. Effet degree-driven

**Attendu** (Chen et al. 2023, Harush & Barzel 2017) :
- F_i ∼ S_i^ω avec ω > 0 : les hubs concentrent le flow
- τ_i ∼ S_i^θ avec θ < 0 : les hubs répondent plus vite
- Pour le modèle SIS : ω ≈ 0 (théorique), θ = -1 (théorique)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

mask = (S > 0) & (F_i > 1e-8) & (tau_i > 1e-8)
log_S = np.log(S[mask])

# ── 1. F_i vs S_i ────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.scatter(S[mask], F_i[mask], alpha=0.6, s=30, color='#2166ac')
omega_fit = np.polyfit(log_S, np.log(F_i[mask]), 1)
S_fit = np.exp(np.linspace(log_S.min(), log_S.max(), 100))
ax.plot(S_fit, np.exp(np.polyval(omega_fit, np.log(S_fit))),
        'r--', lw=2, label=f'ω = {omega_fit[0]:.2f}')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('S_i (degré pondéré)'); ax.set_ylabel('F_i')
ax.set_title('Risk propagation flow vs degré', fontsize=12)
ax.legend(fontsize=11); ax.grid(alpha=0.3)

# ── 2. τ_i vs S_i ────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.scatter(S[mask], tau_i[mask], alpha=0.6, s=30, color='#b2182b')
theta_fit = np.polyfit(log_S, np.log(tau_i[mask]), 1)
ax.plot(S_fit, np.exp(np.polyval(theta_fit, np.log(S_fit))),
        'b--', lw=2, label=f'θ = {theta_fit[0]:.2f}  (théorie : -1)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('S_i (degré pondéré)'); ax.set_ylabel('τ_i')
ax.set_title('Temps de réponse vs degré', fontsize=12)
ax.legend(fontsize=11); ax.grid(alpha=0.3)

# ── 3. Distribution F_i ──────────────────────────────────────────────────
ax = axes[1, 0]
ax.hist(F_i, bins=30, color='#2166ac', edgecolor='white')
ax.set_xlabel('F_i'); ax.set_ylabel('# actifs')
ax.set_title(f'Distribution F_i  (top 10% = {np.sort(F_i)[int(0.9*N):][::-1].sum()/F_i.sum()*100:.1f}% du total)')
ax.grid(alpha=0.3)

# ── 4. PMFG heatmap coloré par F_i ──────────────────────────────────────
ax = axes[1, 1]
# Reordonner par F_i pour voir les hubs
order = np.argsort(F_i)
A_ordered = A_pmfg[np.ix_(order, order)]
sns.heatmap(A_ordered, cmap='YlOrRd', ax=ax, square=True,
            xticklabels=False, yticklabels=False, cbar=True)
ax.set_title('PMFG réordonné par F_i croissant (hubs en bas/droite)')

fig.suptitle(f'SIS Risk Propagation — PMFG, B={B_sis}, R={R_sis}', fontsize=13)
plt.tight_layout(); plt.show()

print(f'\nω (F_i ∼ S_i^ω) = {omega_fit[0]:.3f}  (théorie SIS : 0)')
print(f'θ (τ_i ∼ S_i^θ) = {theta_fit[0]:.3f}  (théorie SIS : -1)')

## 7. Sensibilité aux paramètres R et B

On fait varier R ∈ {0.5, 1, 2} et B ∈ {0.5, 1, 2} et on vérifie que l'effet degree-driven est robuste.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

param_grid = [
    ('R', [(0.5, 1.0), (1.0, 1.0), (2.0, 1.0)], axes[0, 0], axes[0, 1]),
    ('B', [(1.0, 0.5), (1.0, 1.0), (1.0, 2.0)], axes[1, 0], axes[1, 1]),
]
colors = ['#4dac26', '#d01c8b', '#f1b6da']

for param_name, configs, ax_F, ax_tau in param_grid:
    for (R_p, B_p), col in zip(configs, colors):
        # Steady state
        sol_p = solve_ivp(sis_ode, [0, 200], np.full(N, 0.5),
                          args=(A_pmfg, R_p, B_p),
                          method='RK45', rtol=1e-7, atol=1e-9)
        x_p = np.clip(sol_p.y[:, -1], 1e-6, 1 - 1e-6)
        Ax_p = A_pmfg @ x_p
        tau_p = np.log(2) / (B_p + R_p * Ax_p)

        mk = (S > 0) & (tau_p > 0)
        lS = np.log(S[mk])
        th = np.polyfit(lS, np.log(tau_p[mk]), 1)
        S_fit = np.exp(np.linspace(lS.min(), lS.max(), 80))

        ax_tau.scatter(S[mk], tau_p[mk], alpha=0.4, s=20, color=col)
        ax_tau.plot(S_fit, np.exp(np.polyval(th, np.log(S_fit))),
                    '--', lw=2, color=col, label=f'{param_name}={R_p if param_name=="R" else B_p:.1f}  θ={th[0]:.2f}')

        # F_i : recompute J for this param set
        J_p = R_p * np.diag(1 - x_p) @ A_pmfg
        np.fill_diagonal(J_p, -(B_p + R_p * Ax_p))
        G_p_raw = inv(-J_p)
        G_p = np.abs(G_p_raw * x_p[np.newaxis, :] / (x_p[:, np.newaxis] + 1e-10))

        J_subs_p = np.stack([J_p[np.ix_(np.delete(all_idx, i),
                                          np.delete(all_idx, i))] for i in range(N)])
        x_subs_p = np.stack([np.delete(x_p, i) for i in range(N)])
        G_sub_raws_p = np.linalg.inv(-J_subs_p)
        G_subs_p = np.abs(G_sub_raws_p
                          * x_subs_p[:, np.newaxis, :]
                          / (x_subs_p[:, :, np.newaxis] + 1e-10))
        G_col_sum_p = G_p.sum(axis=0)
        G_sub_col_sums_p = G_subs_p.sum(axis=1)
        F_p = np.array([np.mean((np.delete(G_col_sum_p, i) - G_sub_col_sums_p[i])
                                / (np.delete(G_col_sum_p, i) + 1e-10))
                        for i in range(N)])
        mk2 = (S > 0) & (F_p > 1e-8)
        om = np.polyfit(np.log(S[mk2]), np.log(F_p[mk2]), 1)
        ax_F.scatter(S[mk2], F_p[mk2], alpha=0.4, s=20, color=col)
        ax_F.plot(S_fit, np.exp(np.polyval(om, np.log(S_fit))),
                  '--', lw=2, color=col, label=f'{param_name}={R_p if param_name=="R" else B_p:.1f}  ω={om[0]:.2f}')

    for ax in [ax_F, ax_tau]:
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('S_i'); ax.grid(alpha=0.3); ax.legend(fontsize=9)
    ax_F.set_ylabel('F_i'); ax_F.set_title(f'F_i vs S_i — variation {param_name}')
    ax_tau.set_ylabel('τ_i'); ax_tau.set_title(f'τ_i vs S_i — variation {param_name}')

plt.suptitle('Robustesse aux paramètres R, B', fontsize=13)
plt.tight_layout(); plt.show()